# Chapter 1 Lab — What Is Agentic AI?

**Book:** [Agentic AI: Build, Ship, Portfolio](../index.html) | **Chapter:** [What Is Agentic AI?](../part-01-foundations/ch01-what-is-agentic-ai.html)

---

In this lab you will build three tools that make the concepts from Chapter 1 concrete and runnable:

1. **Autonomy Classifier** — Given a system description, classify it on the autonomy spectrum (Script, Chain, Agent, Multi-Agent) using structured LLM output.
2. **Agentic vs Non-Agentic Recommender** — Given a task description, decide whether an agentic or non-agentic approach is appropriate, backed by a decision matrix.
3. **Agent Anatomy Audit** — Given a system description, identify which of the four pillars (Perception, Memory, Reasoning, Action) are present or missing.

> **Prerequisites:** An OpenAI API key stored in a `.env` file (`OPENAI_API_KEY=sk-...`) in the same directory as this notebook or a parent directory.

In [ ]:
# --- Setup ---
# Install required packages (uncomment if needed)
# %pip install openai python-dotenv

import json
import os
from dataclasses import dataclass, asdict
from dotenv import load_dotenv
from openai import OpenAI

load_dotenv()  # loads OPENAI_API_KEY from .env

client = OpenAI()  # uses OPENAI_API_KEY from environment
MODEL = "gpt-4o"   # change to "gpt-4o-mini" for lower cost

print(f"OpenAI client ready  |  Model: {MODEL}")

---
## Section 1 — The Autonomy Spectrum

The chapter defines a four-level **autonomy spectrum**:

| Level | Name | Key Trait |
|-------|------|-----------|
| 0 | **Script** | Deterministic, no decisions |
| 1 | **Chain** | Sequential LLM calls, fixed path |
| 2 | **Agent** | Dynamic decisions, tool use, re-planning |
| 3 | **Multi-Agent** | Delegation and coordination among agents |

We will build an `AutonomyClassifier` that uses the OpenAI API with structured (JSON) output to score any system description on four dimensions of agency and return a classification with reasoning.

In [ ]:
# --- Autonomy Classifier ---

@dataclass
class AutonomyAssessment:
    """Structured result of an autonomy classification."""
    classification: str          # "Script", "Chain", "Agent", "Multi-Agent"
    overall_score: float         # 0.0 – 3.0
    goal_directedness: int       # 0-3
    autonomy: int                # 0-3
    tool_use: int                # 0-3
    adaptive_reasoning: int      # 0-3
    explanation: str

CLASSIFIER_PROMPT = """\
You are an AI systems analyst. Given a description of a software system, evaluate it
on four dimensions of agency, each scored 0-3:

1. Goal-directedness (0=no goals, 1=implicit fixed goal, 2=explicit tracked goal,
   3=multiple dynamic goals)
2. Autonomy (0=no decisions, 1=per-step LLM output, 2=dynamic routing,
   3=full self-direction)
3. Tool use (0=none, 1=hardcoded integrations, 2=model-selected tools,
   3=tool discovery/composition)
4. Adaptive reasoning (0=no adaptation, 1=simple retry, 2=re-planning on failure,
   3=self-reflection and strategy shifts)

Based on the average of these four scores, classify the system:
- 0.0 – 0.7: Script
- 0.8 – 1.5: Chain
- 1.6 – 2.4: Agent
- 2.5 – 3.0: Multi-Agent

Respond with valid JSON matching this exact schema:
{
  "goal_directedness": <int 0-3>,
  "autonomy": <int 0-3>,
  "tool_use": <int 0-3>,
  "adaptive_reasoning": <int 0-3>,
  "classification": "<Script|Chain|Agent|Multi-Agent>",
  "explanation": "<2-3 sentence justification>"
}
"""


def classify_system(description: str) -> AutonomyAssessment:
    """Classify an AI system on the autonomy spectrum using structured LLM output."""
    response = client.chat.completions.create(
        model=MODEL,
        messages=[
            {"role": "system", "content": CLASSIFIER_PROMPT},
            {"role": "user", "content": description},
        ],
        response_format={"type": "json_object"},
        temperature=0.2,
    )
    result = json.loads(response.choices[0].message.content)
    scores = [
        result["goal_directedness"],
        result["autonomy"],
        result["tool_use"],
        result["adaptive_reasoning"],
    ]
    return AutonomyAssessment(
        classification=result["classification"],
        overall_score=round(sum(scores) / len(scores), 2),
        goal_directedness=result["goal_directedness"],
        autonomy=result["autonomy"],
        tool_use=result["tool_use"],
        adaptive_reasoning=result["adaptive_reasoning"],
        explanation=result["explanation"],
    )


def print_assessment(label: str, assessment: AutonomyAssessment):
    """Pretty-print an autonomy assessment."""
    print(f"\n{'=' * 60}")
    print(f"  {label}")
    print(f"{'=' * 60}")
    print(f"  Classification:      {assessment.classification}")
    print(f"  Overall score:       {assessment.overall_score} / 3.0")
    print(f"  ---")
    print(f"  Goal-directedness:   {assessment.goal_directedness}/3")
    print(f"  Autonomy:            {assessment.autonomy}/3")
    print(f"  Tool use:            {assessment.tool_use}/3")
    print(f"  Adaptive reasoning:  {assessment.adaptive_reasoning}/3")
    print(f"  ---")
    print(f"  Explanation: {assessment.explanation}")

print("AutonomyClassifier ready.")

### Test the Autonomy Classifier

Below we run the classifier against five system descriptions that span the entire spectrum — from a simple cron job (Script) to a multi-agent research swarm. The LLM evaluates each description and returns structured scores.

In [ ]:
# --- Test the Autonomy Classifier with 5 system descriptions ---

test_systems = {
    "Nightly ETL Script": (
        "A Python cron job that runs every night at 2 AM. It connects to a PostgreSQL "
        "database, extracts the day's sales records, transforms them into a flat CSV, "
        "and uploads the file to an S3 bucket. If the database connection fails, it "
        "retries three times and then sends an alert email. No LLM is involved."
    ),
    "Summarize-and-Translate Chain": (
        "A pipeline that takes a customer support email, uses GPT-4o to summarize it "
        "into three bullet points, then passes the summary to a second LLM call that "
        "translates it into Spanish, and finally a third call that checks the translation "
        "for tone. The sequence of steps is fixed and always the same."
    ),
    "Kubernetes Ops Agent": (
        "Our system monitors a Kubernetes cluster. Every 5 minutes, it uses an LLM to "
        "analyze recent pod logs and metrics. If it detects an anomaly, it decides "
        "whether to scale up replicas, restart a pod, or alert a human. It checks "
        "whether its action resolved the anomaly and tries a different approach if "
        "the first one didn't work. It uses kubectl, Prometheus API, and PagerDuty."
    ),
    "Coding Assistant (Agent Mode)": (
        "A coding assistant that receives a feature description from a developer. It "
        "reads relevant source files, decides which files to modify, generates code "
        "changes, runs the test suite, reads test output, and iterates — modifying "
        "its approach based on failing tests — until all tests pass or it determines "
        "it cannot make progress. It uses the file system, terminal, linter, and test runner."
    ),
    "Research Swarm": (
        "A multi-agent research system with four specialized agents: a Planner that "
        "decomposes a research question into sub-tasks, a Searcher that queries the "
        "web and academic databases, a Reader that extracts key findings from papers, "
        "and a Writer that synthesizes everything into a report. A Supervisor agent "
        "routes tasks, resolves conflicts between agents, and decides when the "
        "research is complete."
    ),
}

results = {}
for label, description in test_systems.items():
    assessment = classify_system(description)
    results[label] = assessment
    print_assessment(label, assessment)

---
## Section 2 — Agentic vs Non-Agentic

Not every task needs an agent. The chapter warns against the common mistake of adding agency where a simpler approach would be more reliable, cheaper, and faster (the **agency tax**).

Below we build a **recommendation tool** that evaluates a task description against a decision matrix and advises whether an agentic or non-agentic architecture is the right fit. The matrix considers:

| Factor | Favors Non-Agentic | Favors Agentic |
|--------|-------------------|----------------|
| Task variability | Low / predictable | High / open-ended |
| Steps required | Fixed, known | Dynamic, unknown |
| Tool interaction | None or pre-wired | Model-selected |
| Error handling | Retry or fail | Re-plan |
| Latency tolerance | Milliseconds | Seconds to minutes |
| Cost sensitivity | High | Moderate |

In [ ]:
# --- Agentic vs Non-Agentic Recommender ---

@dataclass
class AgenticRecommendation:
    """Result of the agentic-vs-non-agentic analysis."""
    recommendation: str          # "Agentic" or "Non-Agentic"
    confidence: str              # "High", "Medium", "Low"
    task_variability: int        # 1-5 (1=low, 5=high)
    steps_uncertainty: int       # 1-5
    tool_interaction: int        # 1-5
    error_complexity: int        # 1-5
    latency_tolerance: int       # 1-5 (1=needs ms, 5=minutes ok)
    cost_sensitivity: int        # 1-5 (1=very sensitive, 5=not sensitive)
    reasoning: str
    suggested_architecture: str  # "Script", "Chain", "Agent", "Multi-Agent"

RECOMMENDER_PROMPT = """\
You are a software architect specializing in AI systems. Given a task description,
evaluate whether it should be built with an agentic or non-agentic architecture.

Score each factor from 1 to 5:

1. Task variability: How much does the task vary between runs?
   (1=always identical, 5=highly variable/open-ended)
2. Steps uncertainty: How predictable is the sequence of steps?
   (1=completely fixed, 5=impossible to predetermine)
3. Tool interaction: How much external tool use is needed?
   (1=none, 5=many tools, model must choose which)
4. Error complexity: How complex is error recovery?
   (1=retry or fail is fine, 5=requires re-planning and strategy shifts)
5. Latency tolerance: How much time can the task take?
   (1=must be <100ms, 5=minutes or hours are acceptable)
6. Cost sensitivity: How cost-sensitive is this use case?
   (1=every cent matters, 5=cost is not a concern)

Then:
- If the average score is < 2.5, recommend "Non-Agentic"
- If the average score is >= 2.5, recommend "Agentic"
- Suggest a specific architecture: Script, Chain, Agent, or Multi-Agent

Respond with valid JSON:
{
  "task_variability": <int>,
  "steps_uncertainty": <int>,
  "tool_interaction": <int>,
  "error_complexity": <int>,
  "latency_tolerance": <int>,
  "cost_sensitivity": <int>,
  "recommendation": "<Agentic|Non-Agentic>",
  "confidence": "<High|Medium|Low>",
  "suggested_architecture": "<Script|Chain|Agent|Multi-Agent>",
  "reasoning": "<2-3 sentence explanation>"
}
"""


def recommend_approach(task_description: str) -> AgenticRecommendation:
    """Recommend agentic vs non-agentic approach for a given task."""
    response = client.chat.completions.create(
        model=MODEL,
        messages=[
            {"role": "system", "content": RECOMMENDER_PROMPT},
            {"role": "user", "content": task_description},
        ],
        response_format={"type": "json_object"},
        temperature=0.2,
    )
    r = json.loads(response.choices[0].message.content)
    return AgenticRecommendation(
        recommendation=r["recommendation"],
        confidence=r["confidence"],
        task_variability=r["task_variability"],
        steps_uncertainty=r["steps_uncertainty"],
        tool_interaction=r["tool_interaction"],
        error_complexity=r["error_complexity"],
        latency_tolerance=r["latency_tolerance"],
        cost_sensitivity=r["cost_sensitivity"],
        reasoning=r["reasoning"],
        suggested_architecture=r["suggested_architecture"],
    )


def print_recommendation(label: str, rec: AgenticRecommendation):
    """Pretty-print a recommendation."""
    print(f"\n{'=' * 60}")
    print(f"  Task: {label}")
    print(f"{'=' * 60}")
    print(f"  Recommendation:        {rec.recommendation} ({rec.confidence} confidence)")
    print(f"  Suggested architecture: {rec.suggested_architecture}")
    print(f"  ---  Decision Matrix  ---")
    print(f"  Task variability:  {rec.task_variability}/5")
    print(f"  Steps uncertainty: {rec.steps_uncertainty}/5")
    print(f"  Tool interaction:  {rec.tool_interaction}/5")
    print(f"  Error complexity:  {rec.error_complexity}/5")
    print(f"  Latency tolerance: {rec.latency_tolerance}/5")
    print(f"  Cost sensitivity:  {rec.cost_sensitivity}/5")
    avg = round(
        sum([rec.task_variability, rec.steps_uncertainty, rec.tool_interaction,
             rec.error_complexity, rec.latency_tolerance, rec.cost_sensitivity]) / 6, 2
    )
    print(f"  --- Average score: {avg} ---")
    print(f"  Reasoning: {rec.reasoning}")

print("Agentic-vs-Non-Agentic recommender ready.")

### Test the Recommender

We compare three tasks: one clearly non-agentic, one borderline, and one clearly agentic.

In [ ]:
# --- Test the Agentic vs Non-Agentic Recommender ---

tasks = {
    "Generate daily sales PDF report": (
        "Every morning at 7 AM, query yesterday's sales from the database, compute "
        "totals by region, render a PDF report using a fixed template, and email it "
        "to the sales team. The format and recipients never change."
    ),
    "Respond to customer support emails": (
        "Read incoming customer emails, classify the intent, draft a response using "
        "company policy documents, and send it. Some emails are simple FAQ questions; "
        "others require looking up orders, processing refunds, or escalating to a "
        "human. The system should handle the easy cases automatically and flag the rest."
    ),
    "Conduct M&A due diligence on a target company": (
        "Given a set of 50+ contracts, financial statements, and regulatory filings "
        "for an acquisition target, systematically read each document, extract key "
        "terms and risks, cross-reference findings across documents, identify "
        "discrepancies, and produce a structured due diligence report with specific "
        "page references. Adapt the analysis strategy based on what is found."
    ),
}

for label, desc in tasks.items():
    rec = recommend_approach(desc)
    print_recommendation(label, rec)

---
## Section 3 — Agent Anatomy Audit

Chapter 1 defines four pillars that make a system genuinely agentic:

1. **Perception** — How the agent takes in information (user input, tool results, observations)
2. **Memory** — Short-term (conversation/scratchpad) and long-term (vector DB, user profiles)
3. **Reasoning** — Planning, evaluation, decisions (ReAct, Chain-of-Thought, Reflection)
4. **Action** — Tool calls, API requests, outputs that affect the world

The **Agent Anatomy Audit** takes a system description and produces a structured report identifying which pillars are present, which are weak, and which are missing entirely.

In [ ]:
# --- Agent Anatomy Audit ---

@dataclass
class PillarAssessment:
    """Assessment of a single agent pillar."""
    status: str       # "Present", "Weak", "Missing"
    score: int        # 0-3
    evidence: str     # what in the description supports this rating
    recommendation: str  # what to add or improve

@dataclass
class AnatomyAudit:
    """Full anatomy audit of an AI system."""
    system_name: str
    perception: PillarAssessment
    memory: PillarAssessment
    reasoning: PillarAssessment
    action: PillarAssessment
    overall_agentic_readiness: str  # "Not Agentic", "Partially Agentic", "Fully Agentic"
    summary: str

AUDIT_PROMPT = """\
You are an AI architecture auditor. Given a system description, evaluate it against the
four pillars of agent anatomy:

1. Perception — How does the system take in information? (user input parsing, tool
   result interpretation, observation of environment)
   Score: 0=no perception, 1=basic input parsing, 2=structured multi-source input,
   3=rich multimodal perception with preprocessing

2. Memory — Does the system maintain context?
   Score: 0=stateless, 1=single-turn context, 2=short-term working memory (scratchpad),
   3=both short-term and long-term memory (vector DB, user profiles, etc.)

3. Reasoning — How does the system decide what to do next?
   Score: 0=no reasoning (deterministic), 1=single LLM call per input,
   2=multi-step reasoning (CoT, ReAct), 3=advanced (reflection, tree-of-thought, re-planning)

4. Action — How does the system affect the world?
   Score: 0=no external actions, 1=fixed output generation, 2=model-selected tool calls,
   3=rich tool use with error handling and retry

For each pillar, provide:
- status: "Present" (score 2-3), "Weak" (score 1), or "Missing" (score 0)
- score: 0-3
- evidence: what in the description supports your rating
- recommendation: what could be added or improved

Then assess overall agentic readiness:
- "Not Agentic": 2+ pillars are Missing
- "Partially Agentic": 1 pillar is Missing or 2+ are Weak
- "Fully Agentic": all pillars are at least Weak, with 3+ Present

Respond with valid JSON:
{
  "perception": {"status": str, "score": int, "evidence": str, "recommendation": str},
  "memory": {"status": str, "score": int, "evidence": str, "recommendation": str},
  "reasoning": {"status": str, "score": int, "evidence": str, "recommendation": str},
  "action": {"status": str, "score": int, "evidence": str, "recommendation": str},
  "overall_agentic_readiness": str,
  "summary": "<2-3 sentence overall assessment>"
}
"""


def audit_system(system_name: str, description: str) -> AnatomyAudit:
    """Audit a system against the four pillars of agent anatomy."""
    response = client.chat.completions.create(
        model=MODEL,
        messages=[
            {"role": "system", "content": AUDIT_PROMPT},
            {"role": "user", "content": description},
        ],
        response_format={"type": "json_object"},
        temperature=0.2,
    )
    r = json.loads(response.choices[0].message.content)
    return AnatomyAudit(
        system_name=system_name,
        perception=PillarAssessment(**r["perception"]),
        memory=PillarAssessment(**r["memory"]),
        reasoning=PillarAssessment(**r["reasoning"]),
        action=PillarAssessment(**r["action"]),
        overall_agentic_readiness=r["overall_agentic_readiness"],
        summary=r["summary"],
    )


def print_audit(audit: AnatomyAudit):
    """Pretty-print an anatomy audit."""
    print(f"\n{'=' * 64}")
    print(f"  AGENT ANATOMY AUDIT: {audit.system_name}")
    print(f"{'=' * 64}")
    for pillar_name in ["perception", "memory", "reasoning", "action"]:
        p: PillarAssessment = getattr(audit, pillar_name)
        icon = {"Present": "[+]", "Weak": "[~]", "Missing": "[-]"}[p.status]
        print(f"\n  {icon} {pillar_name.upper()} — {p.status} ({p.score}/3)")
        print(f"      Evidence:       {p.evidence}")
        print(f"      Recommendation: {p.recommendation}")
    print(f"\n  {'=' * 60}")
    print(f"  Overall readiness: {audit.overall_agentic_readiness}")
    print(f"  Summary: {audit.summary}")

print("Agent Anatomy Audit ready.")

### Run the Anatomy Audit

We audit two systems: the NovaMart chatbot from the chapter opening (which failed because it lacked agency) and a well-designed coding assistant that has all four pillars.

In [ ]:
# --- Run the Anatomy Audit ---

audit_systems = {
    "NovaMart Customer Chatbot": (
        "A fine-tuned LLM chatbot that answers customer questions about store hours, "
        "return windows, product availability, and shipping estimates. It receives a "
        "question, generates a single response, and has no memory of previous "
        "conversations. It cannot access order databases, initiate returns, or "
        "take any action beyond generating text. Each interaction is independent."
    ),
    "Full-Stack Coding Assistant": (
        "A coding assistant that maintains a scratchpad of the current task state "
        "and a vector database of the codebase. When given a feature request, it "
        "reads relevant files, plans an implementation approach, generates code "
        "changes across multiple files, runs the test suite, interprets failures, "
        "and re-plans its approach. It uses file system access, terminal commands, "
        "a linter, and a test runner. It tracks its goal and stops when tests pass "
        "or when it determines it cannot make further progress."
    ),
}

for name, desc in audit_systems.items():
    audit = audit_system(name, desc)
    print_audit(audit)

---
## Exercises

The following three exercises correspond to the exercises at the end of Chapter 1. Starter code and templates are provided — fill in the missing parts.

### Exercise 1 (Conceptual): Agency Audit of Real-World AI Systems

Choose **five** AI-powered products you use or know about. For each one, classify it on the autonomy spectrum using the `classify_system` function. Then write a one-paragraph assessment of whether its marketing matches its actual architecture.

Here are some systems to get you started (replace or add your own):

| # | System | Your Description |
|---|--------|-----------------|
| 1 | ChatGPT (default conversational mode) | *A reactive chatbot that...* |
| 2 | GitHub Copilot (inline suggestions) | *A code completion tool that...* |
| 3 | Netflix recommendation engine | *A system that...* |
| 4 | Siri / Alexa voice assistant | *A voice-activated system that...* |
| 5 | *(Your choice)* | *...* |

In [ ]:
# --- Exercise 1: Classify 5 real-world AI systems ---
# Replace the descriptions below with your own detailed descriptions.

exercise1_systems = {
    "ChatGPT (default mode)": (
        "A conversational AI that receives a user message, generates a single "
        "response, and waits for the next message. It maintains conversation "
        "history within a session but has no tools, no goal tracking, and no "
        "ability to take actions beyond generating text."
        # TODO: expand this description based on your experience
    ),
    "GitHub Copilot (inline suggestions)": (
        "TODO: Write a 2-3 sentence description of how this system works."
    ),
    "Netflix Recommendation Engine": (
        "TODO: Write a 2-3 sentence description of how this system works."
    ),
    "Siri / Alexa": (
        "TODO: Write a 2-3 sentence description of how this system works."
    ),
    "Your Choice Here": (
        "TODO: Replace with a system you use and describe it."
    ),
}

# Uncomment below once you have filled in the descriptions:
# for label, description in exercise1_systems.items():
#     assessment = classify_system(description)
#     print_assessment(label, assessment)

### Exercise 2 (Coding): Batch Classifier with Radar Chart

Extend the Autonomy Classifier to process multiple systems and visualize their agency scores as **radar charts** using `matplotlib`. Each system gets a four-axis radar (Goal-directedness, Autonomy, Tool Use, Adaptive Reasoning).

Starter code is provided below. Complete the `plot_radar_charts` function.

In [ ]:
# --- Exercise 2: Radar chart visualization ---
# %pip install matplotlib  # uncomment if needed

import numpy as np
import matplotlib.pyplot as plt


def plot_radar_charts(assessments: dict[str, AutonomyAssessment]):
    """
    Plot a radar chart for each assessed system.

    Each chart has four axes: Goal-directedness, Autonomy, Tool Use, Adaptive Reasoning.
    Scores range from 0 to 3.
    """
    categories = ["Goal-\ndirectedness", "Autonomy", "Tool Use", "Adaptive\nReasoning"]
    num_vars = len(categories)

    # Compute angle for each axis
    angles = np.linspace(0, 2 * np.pi, num_vars, endpoint=False).tolist()
    angles += angles[:1]  # close the polygon

    fig, axes = plt.subplots(
        1, len(assessments), figsize=(5 * len(assessments), 5),
        subplot_kw=dict(polar=True),
    )
    if len(assessments) == 1:
        axes = [axes]

    colors = ["#0F6E56", "#185FA5", "#534AB7", "#993C1D", "#7B6D00"]

    for ax, (label, assessment), color in zip(axes, assessments.items(), colors):
        values = [
            assessment.goal_directedness,
            assessment.autonomy,
            assessment.tool_use,
            assessment.adaptive_reasoning,
        ]
        values += values[:1]  # close the polygon

        ax.set_ylim(0, 3)
        ax.set_yticks([1, 2, 3])
        ax.set_yticklabels(["1", "2", "3"], fontsize=8, color="gray")
        ax.set_xticks(angles[:-1])
        ax.set_xticklabels(categories, fontsize=9)

        ax.plot(angles, values, "o-", linewidth=2, color=color)
        ax.fill(angles, values, alpha=0.2, color=color)
        ax.set_title(
            f"{label}\n({assessment.classification} — {assessment.overall_score}/3.0)",
            fontsize=11, fontweight="bold", pad=20,
        )

    plt.tight_layout()
    plt.show()


# TODO: Use the `results` dict from the Autonomy Classifier test above,
# or re-run classify_system on your own system descriptions, then call:
#
# plot_radar_charts(results)
#
# Uncomment the line below once you have results:
# plot_radar_charts(results)

### Exercise 3 (Design): Agency Migration Plan

Think of a real system at your organization (or a hypothetical one) that is currently a **Script** or **Chain** but would benefit from being more agentic. Fill in the architecture decision record (ADR) template below.

Use the `audit_system` function to generate an anatomy audit of the current system, then design the target architecture that addresses the gaps.

#### ADR Template: Agency Migration Plan

**System name:** *(e.g., "Invoice Processing Pipeline")*

**Current state:** Script / Chain *(circle one)*

**Current description:**
> *(Describe the system as it exists today in 3-5 sentences.)*

---

**1. What specific capabilities does agency unlock?**
- *(Capability 1)*
- *(Capability 2)*
- *(Capability 3)*

**2. What is the minimum viable level of agency needed?**
> *(Agent or Multi-Agent? Which of the four pillars must be added or strengthened?)*

**3. What new failure modes must be mitigated?**
| Failure Mode | Likelihood | Mitigation |
|-------------|-----------|------------|
| *(e.g., Infinite loop)* | *(High/Med/Low)* | *(e.g., Step count limit)* |
| | | |
| | | |

**4. How will you test and observe the agentic behavior?**
- *(Testing strategy)*
- *(Observability: traces, logs, metrics)*

**5. What is the rollback plan?**
> *(How do you revert to the current system if the agent performs worse?)*

---

**Decision:** Migrate / Do Not Migrate *(circle one)*

**Rationale:** *(1-2 sentences)*

In [ ]:
# --- Exercise 3: Audit the current system to identify gaps ---
# Replace the description below with your own system, then run the audit.

current_system_name = "Invoice Processing Pipeline"  # TODO: replace
current_system_description = (
    "TODO: Describe the current system in 3-5 sentences. "
    "What does it do? How does it work? What are its limitations?"
)

# Uncomment below once you have filled in the description:
# audit = audit_system(current_system_name, current_system_description)
# print_audit(audit)
# print("\nUse the audit results above to fill in the ADR template.")